# PS4 v4: 3D hand tracking (Colab)
**Before you start**
1. Runtime → Change runtime type → **A100** (T4 also works, ~2–3× slower).
2. Upload **`ps4v4.zip`** to **`MyDrive/mytronD/`** (same folder as the clips).
3. Clips are named `hand_tracking_1.mp4`, `hand_tracking_2.mp4`, … in `MyDrive/mytronD/`.

Run the cells top to bottom. Results appear in **`MyDrive/mytronD/ps4v4_results/<clip>/`** as each clip finishes.

*Data rule: the outputs contain footage. Keep them private and delete all copies after the event.*

In [ ]:
# CELL 1: install (~3 min)
!pip -q install ultralytics "smplx==0.1.28" timm einops roma scikit-image huggingface_hub pyyaml setuptools wheel
!pip -q install --no-deps git+https://github.com/warmshao/WiLoR-mini
!pip -q install --no-build-isolation --no-deps git+https://github.com/mattloper/chumpy
!pip -q install git+https://github.com/cvg/GeoCalib
import importlib.util, pathlib
_p = pathlib.Path(importlib.util.find_spec("chumpy").submodule_search_locations[0])
for _f in _p.glob("*.py"):                      # chumpy (MANO loading) breaks on new Python/NumPy
    _s = _f.read_text(); _o = _s
    _s = _s.replace("from numpy import bool, int, float, complex, object, unicode, str, nan, inf", "from numpy import nan, inf")
    _s = _s.replace("inspect.getargspec", "inspect.getfullargspec")
    if _s != _o: _f.write_text(_s)
import chumpy, torch
print("install OK | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU!")

In [ ]:
# CELL 2: mount Drive and unpack the code
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil, os
DRIVE_DIR = "/content/drive/MyDrive/mytronD"
ZIP = f"{DRIVE_DIR}/ps4v4.zip"
assert os.path.exists(ZIP), "upload ps4v4.zip to MyDrive/mytronD first"
shutil.rmtree("/content/ps4v4", ignore_errors=True)
zipfile.ZipFile(ZIP).extractall("/content")
!cd /content/ps4v4 && python tests/test_synthetic.py 2>&1 | tail -3      # 1-min self-test, must end with ALL CHECKS PASSED
# download the model files ONCE here; parallel clips downloading at the same time crash each other
!cd /content/ps4v4 && python -c "from ps4v4.models.wilor_loader import load_wilor; load_wilor('cuda'); print('WiLoR weights OK')"
!python -c "from geocalib import GeoCalib; GeoCalib(); print('GeoCalib weights OK')"
!python -c "from transformers import CLIPModel, CLIPProcessor as P; CLIPModel.from_pretrained('openai/clip-vit-base-patch32'); P.from_pretrained('openai/clip-vit-base-patch32'); print('CLIP weights OK')"

In [ ]:
# CELL 3: run every hand_tracking_<n>.mp4 (results land in Drive as each clip finishes)
import glob, os, re, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed

DRIVE_DIR = "/content/drive/MyDrive/mytronD"
RESULTS   = f"{DRIVE_DIR}/ps4v4_results"
WORK      = "/content/ps4v4_work"          # fast local cache; finished files are copied to Drive
WORKERS   = 3                              # clips in parallel (A100: 3, T4: 1)
MAX_SECONDS = None                         # e.g. 10 for a quick check
ARGS = ["camera.use_geocalib=true"]        # any config overrides, e.g. "assign.accept_bias=0.3"
KEEP = ["render.mp4", "hands.jsonl", "robot_wrist_gripper.csv", "stats.json", "camera.json", "tracklets.csv", "hands_mesh.npz", "quality.json",
        "tracklets.pkl", "probe.pkl", "head_rot.pkl"]   # .pkl = numbers only (no footage): lets the decision step be re-tuned offline

clips = sorted([p for p in glob.glob(f"{DRIVE_DIR}/hand_tracking_*.mp4")
                if re.fullmatch(r"hand_tracking_\d+\.mp4", os.path.basename(p))],
               key=lambda p: int(re.findall(r"\d+", os.path.basename(p))[0]))
print(len(clips), "clips:", [os.path.basename(c) for c in clips])
os.makedirs(RESULTS, exist_ok=True); os.makedirs(WORK, exist_ok=True); os.makedirs("/content/videos", exist_ok=True)

def run_one(clip):
    name = os.path.splitext(os.path.basename(clip))[0]
    out = f"{RESULTS}/{name}"
    if os.path.exists(f"{out}/render.mp4"):
        return name, "already done (delete its results folder to re-run)"
    local = f"/content/videos/{name}.mp4"
    if not os.path.exists(local):
        shutil.copy(clip, local)                      # reading from local disk is much faster than Drive
    sets = [f"paths.cache_root={WORK}", f"max_seconds={'null' if MAX_SECONDS is None else MAX_SECONDS}", *ARGS]
    t0 = time.time()
    log = f"{WORK}/{name}.log"
    with open(log, "w") as fh:
        rc = subprocess.run([sys.executable, "run.py", "--clip", local, "--set", *sets],
                            cwd="/content/ps4v4", stdout=fh, stderr=subprocess.STDOUT).returncode
    os.makedirs(out, exist_ok=True)
    shutil.copy(log, f"{out}/run.log")
    if rc != 0:
        return name, f"FAILED (see {out}/run.log)"
    for f in KEEP:
        src = f"{WORK}/{name}/{f}"
        if os.path.exists(src):
            shutil.copy(src, f"{out}/{f}")
    return name, f"done in {(time.time() - t0) / 60:.1f} min -> {out}"

t_all = time.time()
with ThreadPoolExecutor(WORKERS) as ex:
    for fut in as_completed([ex.submit(run_one, c) for c in clips]):
        name, msg = fut.result()
        print(f"[{(time.time() - t_all) / 60:5.1f} min] {name}: {msg}", flush=True)
print("all done")

In [ ]:
# CELL 4: summary table + one render
import json, glob, os
import pandas as pd
from IPython.display import Video, display
rows = []
for d in sorted(glob.glob(f"{RESULTS}/hand_tracking_*")):
    p = f"{d}/stats.json"
    if not os.path.exists(p):
        continue
    s = json.load(open(p))
    rows.append({"clip": os.path.basename(d), "focal": round(s["camera"]["fx"]), "z_ref_m": s.get("z_ref_m"),
                 **{f"{k}_tracklets": v for k, v in s.get("tracklets", {}).items()},
                 **{f"{sd}_observed": s[sd]["share_observed"] for sd in ("left", "right")},
                 **{f"{sd}_estimated": s[sd]["share_estimated_out_of_view"] for sd in ("left", "right")},
                 "minutes": round(sum(s["seconds"].values()) / 60, 1)})
display(pd.DataFrame(rows))
SHOW = "hand_tracking_2"                        # which render to play here
display(Video(f"{RESULTS}/{SHOW}/render.mp4", embed=True, width=960))

### Tuning without re-running everything
Every tracklet's evidence and decision is in `tracklets.csv` (columns `hand`, `side`, `label`, `reasons`).
Rejected tracklets show in the render as grey boxes with the reason (`static`, `far x2.4`, `arm away`, `kp/box`, `side taken`, `short`).
Changing only `assign.*` settings re-runs just the decision step and later steps (a minute or two per clip), as long as `/content/ps4v4_work` still exists:

In [ ]:
# CELL 5 (optional): re-decide one clip with different assign settings, e.g. stricter
CLIP = "hand_tracking_4"
!cd /content/ps4v4 && python run.py --clip /content/videos/{CLIP}.mp4 --set paths.cache_root={WORK} camera.use_geocalib=true assign.accept_bias=0.3
!cp {WORK}/{CLIP}/render.mp4 {WORK}/{CLIP}/tracklets.csv {WORK}/{CLIP}/stats.json {RESULTS}/{CLIP}/